# 2D Burgers with domain decomposition
Demonstrate the domain decomposition formulation applied to the 2D Burgers equation.

In [ ]:
import sys
with open("./../../../PATHS.txt") as file:
  paths = file.read().splitlines()
sys.path.extend(paths)

In [ ]:
from dd_nm_rom import env
env.set(
  backend="numpy",
  device="cpu",
  device_idx=0,
  nb_threads=4,
  epsilon=1e-10,
  floatx="float64",
  seed=0
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from matplotlib import cm

In [ ]:
from dd_nm_rom.postproc import *
from dd_nm_rom import fom as fom_mod
from dd_nm_rom import field as field_mod
from dd_nm_rom.elements import mesh as mesh_mod

In [ ]:
# plt.rc("font", size=20)
# plt.rcParams["text.usetex"] = False
# plt.rcParams["lines.markersize"] = 10

## 2D Burgers Equation
The 2D steady-state viscous Burgers Equation on $\Omega = [a, b] \times [c, d]\subset \real^2$ with viscosity $\nu>0$ and inhomogeneous Dirichlet boundary conditions is given by
\begin{align*}
    u \pdx u + v \pdy u &= \nu \left(\pdxx u + \pdyy u\right), \qquad (x, y)\in\Omega, \\
    u \pdx v + v \pdy v &= \nu \left(\pdxx v + \pdyy v\right), \qquad (x, y)\in\Omega,\\
    u(x, y) &= u_D(x, y), \qquad (x, y) \in \partial \Omega, \\
    v(x, y) &= v_D(x, y), \qquad (x, y) \in \partial \Omega. \\
\end{align*}
The domain is discretized uniformly with $n_x+2$ grid points in the $x$-direction and $n_y+2$ grid points in the $y$-direction, resulting in grid points $(x_i, y_j)$ where 
\begin{align*}
x_i = a + i h_x, \qquad i = 0, \dots, n_x+1, \\
y_j = c + j h_y, \qquad j = 0, \dots, n_y+1,
\end{align*}
where $h_x = (b-a)/(n_x+1)$ and $h_y = (d-c)/(n_y+1)$.
The solutions $u, v$ on the grid points are denoted $u_{ij} \approx u(x_i, y_j)$ and $v_{ij} \approx v(x_i, y_j)$. 
The PDE is then discretized using centered finite differences for the first and second derivative terms. The fully discretized system is given by
\begin{align*}
\bff_u(\bu, \bv) &= \bzero, \\
\bff_v(\bu, \bv) &= \bzero,
\end{align*}
where
\begin{align*}
\bu &=
\begin{bmatrix}
\bu^{[1]} \\ \vdots \\ \bu^{[n_y]}
\end{bmatrix} \in \real^{n_xn_y}, \qquad 
\bu^{[j]} = 
\begin{bmatrix}
u_{1, j} \\ \vdots \\ u_{n_x, j}
\end{bmatrix} \in \real^{n_x}, \quad j = 1, \dots, n_y,\\
\bv &=
\begin{bmatrix}
\bv^{[1]} \\ \vdots \\ \bv^{[n_y]}
\end{bmatrix} \in \real^{n_xn_y}, \qquad 
\bv^{[j]} = 
\begin{bmatrix}
v_{1, j} \\ \vdots \\ v_{n_x, j}
\end{bmatrix}\in \real^{n_x}, \quad j = 1, \dots, n_y,\\
 \bff_u(\bu, \bv)&= \bu \odot(\bB_x \bu - \bb_{ux1}) + \bv \odot(\bB_y \bu - \bb_{uy1}) \\
 &\quad + \bC_x \bu + \bb_{ux2} + \bC_y \bu + \bb_{uy2}, \\
  \bff_v(\bu, \bv)&= \bu \odot(\bB_x \bv - \bb_{vx1}) + \bv \odot(\bB_y \bv - \bb_{vy1}) \\
 &\quad + \bC_x \bv + \bb_{vx2} + \bC_y \bv + \bb_{vy2}, \\
\end{align*}
and where 
\begin{align*}
\bB_x &= -\frac{1}{2h_x} \left(\bI_{n_y} \otimes \tbB_x\right)\in \real^{n_xn_y \times n_x n_y}, \qquad
\tbB_x = 
\begin{bmatrix}
0 & 1 \\
-1 & \ddots & 1 \\
& -1 & 0
\end{bmatrix} \in \real^{n_x \times n_x}, \\
\bB_y &= -\frac{1}{2h_y}\left(\tbB_y \otimes \bI_{n_x}\right) \in \real^{n_xn_y \times n_x n_y} \qquad
\tbB_y = 
\begin{bmatrix}
0 & 1 \\
-1 & \ddots & 1 \\
& -1 & 0
\end{bmatrix} \in \real^{n_y \times n_y}, \\
\bC_x &= \frac{\nu}{h_x^2}\left(\bI_{n_y} \otimes \tbC_x\right) \in \real^{n_xn_y \times n_x n_y}, \qquad
\tbC_x = 
\begin{bmatrix}
-2 & 1 \\
1 & \ddots \\
& 1 & -2 \\
\end{bmatrix} \in \real^{n_x \times n_x}, \\
\bC_y &= \frac{\nu}{h_y^2} \left(\tbC_y \otimes \bI_{n_x}\right) \in \real^{n_xn_y \times n_x n_y}, \qquad
\tbC_y =
\begin{bmatrix}
-2 & 1 \\
1 & \ddots \\
& 1 & -2 \\
\end{bmatrix} \in \real^{n_y \times n_y}, \\
\bb_{ux1} &= -\frac{1}{2h_x} (\bb_{ux\ell} - \bb_{uxr}), \qquad
\bb_{uy1} = -\frac{1}{2h_y}(\bb_{uy\ell} - \bb_{uyr}), \\
\bb_{ux2} &= \frac{\nu}{h_x^2}(\bb_{ux\ell} + \bb_{uxr}), \qquad
\bb_{uy2} = \frac{\nu}{h_y^2} (\bb_{uy\ell} + \bb_{uyr}), \\
\bb_{vx1} &= -\frac{1}{2h_x} (\bb_{vx\ell} - \bb_{vxr}), \qquad
\bb_{vy1} = -\frac{1}{2h_y}(\bb_{vy\ell} - \bb_{vyr}), \\
\bb_{vx2} &= \frac{\nu}{h_x^2}(\bb_{vx\ell} + \bb_{vxr}), \qquad
\bb_{vy2} = \frac{\nu}{h_y^2} (\bb_{vy\ell} + \bb_{vyr}), \\
\bb_{ux\ell} &= 
\begin{bmatrix}
u_D(x_0, y_1) \\ \vdots \\ u_D(x_0, y_{n_y})
\end{bmatrix} \otimes 
\begin{bmatrix}
1 \\ 0 \\ \vdots \\ 0
\end{bmatrix}_{n_x \times 1} \in \real^{n_x n_y}, \qquad
\bb_{uxr} = 
\begin{bmatrix}
u_D(x_{n_x+1}, y_1) \\ \vdots \\ u_D(x_{n_x+1}, y_{n_y})
\end{bmatrix} \otimes 
\begin{bmatrix}
0 \\ \vdots \\ 0 \\ 1
\end{bmatrix}_{n_x \times 1} \in \real^{n_x n_y}, \\
\bb_{uyb} &= 
\begin{bmatrix}
1 \\ 0 \\ \vdots \\ 0
\end{bmatrix}_{n_y \times 1} 
\otimes 
\begin{bmatrix}
u_D(x_1, y_0) \\ \vdots \\ u_D(x_{n_x}, y_0)
\end{bmatrix} \in \real^{n_x n_y} \qquad
\bb_{uyt} = 
\begin{bmatrix}
0 \\ \vdots \\ 0 \\ 1
\end{bmatrix}_{n_y \times 1} 
\otimes 
\begin{bmatrix}
u_D(x_1, y_{n_y+1}) \\ \vdots \\ u_D(x_{n_x}, y_{n_y+1})
\end{bmatrix} \in \real^{n_x n_y} \\
\bb_{vx\ell} &= 
\begin{bmatrix}
v_D(x_0, y_1) \\ \vdots \\ v_D(x_0, y_{n_y})
\end{bmatrix} \otimes 
\begin{bmatrix}
1 \\ 0 \\ \vdots \\ 0
\end{bmatrix}_{n_x \times 1} \in \real^{n_x n_y}, \qquad
\bb_{vxr} = 
\begin{bmatrix}
v_D(x_{n_x+1}, y_1) \\ \vdots \\ v_D(x_{n_x+1}, y_{n_y})
\end{bmatrix} \otimes 
\begin{bmatrix}
0 \\ \vdots \\ 0 \\ 1
\end{bmatrix}_{n_x \times 1} \in \real^{n_x n_y}, \\
\bb_{vyb} &= 
\begin{bmatrix}
1 \\ 0 \\ \vdots \\ 0
\end{bmatrix}_{n_y \times 1} 
\otimes 
\begin{bmatrix}
v_D(x_1, y_0) \\ \vdots \\ v_D(x_{n_x}, y_0)
\end{bmatrix} \in \real^{n_x n_y} \qquad
\bb_{vyt} = 
\begin{bmatrix}
0 \\ \vdots \\ 0 \\ 1
\end{bmatrix}_{n_y \times 1} 
\otimes 
\begin{bmatrix}
v_D(x_1, y_{n_y+1}) \\ \vdots \\ v_D(x_{n_x}, y_{n_y+1})
\end{bmatrix} \in \real^{n_x n_y} 
\end{align*}

In [ ]:
nx, ny = 480, 24
x_lim = [-1.0, 1.0]
y_lim = [0.0, 0.05]
a_lim = [1.0, 10000.0]
k_lim = [5.0, 25.0]
a1 = 1e4
lam = 5.0
mu = np.array([a1, lam])
viscosity = 1e-1

# fig_dir = f"/Users/zanardi1/Workspace/Codes/DD-NM-ROM/run/figures/steady/nx_{nx}_ny_{ny}_mu_{viscosity}/fom/"
fig_dir = f"/usr/workspace/zanardi1/Codes/DD-NM-ROM/run/figures/steady/nx_{nx}_ny_{ny}_mu_{viscosity}/fom/"
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
mesh_mono = mesh_mod.MeshMono(
  nx=nx,
  ny=ny,
  x_lim=x_lim,
  y_lim=y_lim
)
mesh_mono.build()
X, Y = mesh_mono.grid

In [ ]:
field = field_mod.Burgers2DExact(
  mesh=mesh_mono,
  nu=viscosity,
  a_lim=a_lim,
  k_lim=k_lim
)
field.set_params(mu)
fom = fom_mod.Burgers2D(
  nu=viscosity,
  mesh=mesh_mono
)
fom.build(field)

In [ ]:
uv, res, converged = fom.solve(tol=1e-8, maxit=20, stepsize_min=1e-20, verbose=True)
print("RUNTIME:", fom.runtime)

In [ ]:
# compute exact u and v on grid
Uex, Vex = field.u(X, Y), field.v(X, Y)
# plot FD u and v
U = uv["u"].reshape(ny, nx)
V = uv["v"].reshape(ny, nx)

In [ ]:
plot_kwargs = dict(
  x=X,
  y=Y,
  lim=None,
  figsize=(12, 4),
  cmap=cm.jet,
  save=True,
  show=True
)
plot_field(
  z=Uex,
  label="$u_{ex}$",
  filename=fig_dir + "/u_ex.png",
  **plot_kwargs
)
plot_field(
  z=Vex,
  label="$v_{ex}$",
  filename=fig_dir + "/v_ex.png",
  **plot_kwargs
)
plot_field(
  z=U,
  label="$u$",
  filename=fig_dir + "/u.png",
  **plot_kwargs
)
plot_field(
  z=V,
  label="$v$",
  filename=fig_dir + "/v.png",
  **plot_kwargs
)
plot_field(
  z=np.abs(U-Uex)/np.linalg.norm(Uex),
  label="$u_{err}$",
  filename=fig_dir + "/u_err.png",
  **plot_kwargs
)
plot_field(
  z=np.abs(V-Vex)/np.linalg.norm(Vex),
  label="$v_{err}$",
  filename=fig_dir + "/v_err.png",
  **plot_kwargs
)

In [ ]:
# compute relative error between exact and FD solutions
u_rel_err = np.linalg.norm(U-Uex)/np.linalg.norm(Uex)
v_rel_err = np.linalg.norm(V-Vex)/np.linalg.norm(Vex)
print(f"u relative error = {u_rel_err:1.4e}")
print(f"v relative error = {v_rel_err:1.4e}")

In [ ]:
# compute relative error between exact and FD solutions
u_rel_err = 100*np.mean(np.abs(U-Uex)/(np.abs(Uex)+1e-7))
v_rel_err = 100*np.mean(np.abs(V-Vex)/(np.abs(Vex)+1e-7))
print(f"u relative error = {u_rel_err:1.4e}")
print(f"v relative error = {v_rel_err:1.4e}")

## Domain-decomposition
The full-order model (FOM) can expressed as a parametrized system of nonlinear algebraic equations
\begin{equation}\label{eq:fom_residual}
    \br(\bx; \bmu) = \bzero,
\end{equation}
where $\br: \real^{N_x}\times \cD \to \real^{N_x}$ denotes the residual and is nonlinear in (at least) its first argument, $\bmu \in \cD \subset \real^{N_\mu}$
denotes the FOM parameters, and $\bx:\cD \to \real^{N_x}$ denotes the state.
For notational simplicity, the dependence on $\bmu$ is suppressed until needed.
Next consider a decomposition of the system of the FOM into $n_\Omega \leq N_x$ algebraic subdomains such that the residual satisfies
\begin{equation}\label{eq:fom_dd_residual}
  \br(\bw) = \sum_{i=1}^{n_\Omega} \left(\bP_i^r\right)^T \br_i(\bP_i^\Omega \bw, \bP_i^\Gamma \bw), \qquad \forall \; \bw  \in \real^{N_x},
\end{equation}
where $\br_i: \real^{N_i^\Omega} \times \real^{N_i^\Gamma} \to \real^{N_i^r}$ denotes the residual on the $i$th subdomain,
$\bP_i^r \in \set{0, 1}^{N_i^r \times N_x}$ denotes the $i$th residual sampling matrix,
$\bP_i^\Omega \in \set{0, 1}^{N_i^\Omega \times N_x}$ denotes the $i$th interior-state sampling matrix, and
$\bP_i^\Gamma \in \set{0, 1}^{N_i^\Gamma \times N_x}$ denotes the $i$th interface-state sampling matrix.
The variables 
$$
\bw_i^\Omega := \bP_i^\Omega \bw, \qquad
\bw_i^\Gamma := \bP_i^\Gamma \bw,
$$
are also referred to as the interior and interface states on the $i$th subdomain, respectively. 

Next the domain-decomposition formulation is illustrated on a coarse domain. The subdomain grids are visualized for each subdomain, including each of the interior and interface states and ports.

In [ ]:
nx_intr = 3
ny_intr = 3
lx_sub = 0.5
ly_sub = 0.5
x0 = 0.0
y0 = 0.0
n_sub_x = 2
n_sub_y = 2

In [ ]:
mesh = mesh_mod.MeshDD(
  nx_intr=nx_intr,
  ny_intr=ny_intr,
  lx_sub=lx_sub,
  ly_sub=ly_sub,
  x0=x0,
  y0=y0,
  n_sub_x=n_sub_x,
  n_sub_y=n_sub_y
)
mesh.build()
field = field_mod.Burgers2DExact(
  mesh=mesh,
  nu=viscosity,
  a_lim=a_lim,
  k_lim=k_lim
)
field.set_params(mu)
fom = fom_mod.Burgers2D(
  nu=viscosity,
  mesh=mesh
)
fom.build(field)
dd_fom = fom_mod.DDBurgers2D(fom)
dd_fom.build()

In [ ]:
X, Y = mesh.grid
xx, yy = X.flatten(), Y.flatten()

# plot interior states
plt.figure(figsize=(10,10))
m = ["o", "^", "d", "P", ">", "*", "1", "2", "3", "4", "8"]
size = 300
for i, s in enumerate(dd_fom.subdomains):
    indices = s.elem_states["interior"].nodes_state
    plt.scatter(xx[indices], yy[indices], marker=m[i%len(m)], s=size)
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
# plt.legend()
plt.title("Interior States")
plt.show()

In [ ]:
# plot interface states
plt.figure(figsize=(10,10))
for i, s in enumerate(dd_fom.subdomains):
    indices = s.elem_states["interface"].nodes_state
    plt.scatter(xx[indices], yy[indices], s=size, marker=m[i%len(m)], label=f"$\Omega_{i}$")
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
# plt.legend()
plt.title("Interface States")
plt.show()

In [ ]:
# plot ports
plt.figure(figsize=(10,10))
for i, port in dd_fom.dd_indices.port_to_nodes.items():
    plt.scatter(xx[port], yy[port], s=size, marker=m[i%len(m)], label=f"{i}")
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
plt.title("Ports")
file = fig_dir + "/dd_ports.png"
plt.savefig(file, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# plot residual indices
plt.figure(figsize=(10,10))
for i, s in enumerate(dd_fom.subdomains):
    indices = s.elem_states["interior"].nodes_res
    plt.scatter(xx[indices], yy[indices], s=size, marker="o", label=f"$\Omega_{i}$")
plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
file = fig_dir + "/dd_residuals.png"
plt.savefig(file, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# plot subdomain indices
plt.figure(figsize=(10,10))
colors = ["blue", "orange", "green", "red"]
for i, s in enumerate(dd_fom.subdomains):
    interior = s.elem_states["interior"].nodes_state
    interface = s.elem_states["interface"].nodes_state
    xx_i = np.concatenate((xx[interior], xx[interface]))
    yy_i = np.concatenate((yy[interior], yy[interface]))
    plt.scatter(xx_i, yy_i, s=size, marker=m[i%len(m)], label=f"$\Omega_{i}$")

plt.tick_params(left = False, right = False , labelleft = False ,
                labelbottom = False, bottom = False)
file = fig_dir + "/dd_states.png"
plt.savefig(file, bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
# checks that constraint matrices are computed correctly
c = np.zeros(dd_fom.n_constraints)
vec = np.random.rand(2*mesh.nxy)
for s in dd_fom.subdomains:
    interface = s.elem_states["interface"].nodes_state
    c += s.cmat["interface"]@np.concatenate([vec[interface],vec[mesh.nxy+interface]])
print("||sum(A[i] x[i])||=", np.linalg.norm(c))

In [ ]:
# plot sparsity of jacobian
_, jac = dd_fom.res_jac(np.random.rand(dd_fom.get_ndof()))[:2]
plt.figure(figsize=(10, 10))
plt.spy(jac)
plt.show()

In [ ]:
# plot sparsity of constraint matrix
plt.figure(figsize=(10, 10))
plt.spy(dd_fom.subdomains[0].cmat["interface"])
plt.show()

# Compute state solutions u and v using DD model

In [ ]:
mesh_dd = mesh_mod.MeshDD(
  **mesh_mono.get_config_dd(n_sub_x=2, n_sub_y=1)
)
mesh_dd.build()
X, Y = mesh_dd.grid

In [ ]:
field = field_mod.Burgers2DExact(
  mesh=mesh_dd,
  nu=viscosity,
  a_lim=a_lim,
  k_lim=k_lim
)
field.set_params(mu)
fom = fom_mod.Burgers2D(
  nu=viscosity,
  mesh=mesh_dd
)
fom.build(field)

In [ ]:
# generate Burgers FOM on coarse grid for visualization
uv, res, converged = fom.solve(tol=1e-8, maxit=20, stepsize_min=1e-20, verbose=True)
print("RUNTIME:", fom.runtime)

In [ ]:
# compute exact u and v on grid
Uex, Vex = field.u(X, Y), field.v(X, Y)
# plot FD u and v
U = uv["u"].reshape(ny, nx)
V = uv["v"].reshape(ny, nx)

In [ ]:
plot_kwargs["save"] = False
plot_kwargs["x"] = X
plot_kwargs["y"] = Y
plot_field(
  z=Uex,
  label="$u_{ex}$",
  **plot_kwargs
)
plot_field(
  z=Vex,
  label="$v_{ex}$",
  **plot_kwargs
)
plot_field(
  z=U,
  label="$u$",
  **plot_kwargs
)
plot_field(
  z=V,
  label="$v$",
  **plot_kwargs
)
plot_field(
  z=np.abs(U-Uex)/np.linalg.norm(Uex),
  label="$u_{err}$",
  **plot_kwargs
)
plot_field(
  z=np.abs(V-Vex)/np.linalg.norm(Vex),
  label="$v_{err}$",
  **plot_kwargs
)

In [ ]:
# compute DD model
dd_fom_s = fom_mod.DDBurgers2D(fom, constraint_type="strong", scaling=-1)
dd_fom_s.build()

In [ ]:
uv_dd_s, lambdas, res_dd, converged = dd_fom_s.solve(tol=1e-8, maxit=50, stepsize_min=1e-20, verbose=True)
print("RUNTIME:", dd_fom_s.runtime)

In [ ]:
# plot FD u and v
U_dd_s = uv_dd_s["res"]["u"].reshape(ny, nx)
V_dd_s = uv_dd_s["res"]["v"].reshape(ny, nx)

In [ ]:
plot_kwargs["save"] = True
plot_field(
  z=U_dd_s,
  label="$u_{dd-s}$",
  filename=fig_dir + "/u_dd_strong.png",
  **plot_kwargs
)
plot_field(
  z=V_dd_s,
  label="$v_{dd-s}$",
  filename=fig_dir + "/v_dd_strong.png",
  **plot_kwargs
)

In [ ]:
# compute DD model
dd_fom_w = fom_mod.DDBurgers2D(fom, constraint_type="weak", n_constraints_weak=47, scaling=-1)
dd_fom_w.build()

uv_dd_w, lambdas_w, res_dd, converged = dd_fom_w.solve(tol=1e-8, maxit=50, stepsize_min=1e-20, verbose=True)
print("RUNTIME:", dd_fom_w.runtime)

In [ ]:
error = 0.0
for j in range(mesh_dd.n_sub):
    num = np.sum(np.square(uv_dd_s["interior"]["u"][j]-uv_dd_w["interior"]["u"][j])) +\
          np.sum(np.square(uv_dd_s["interior"]["v"][j]-uv_dd_w["interior"]["v"][j])) +\
          np.sum(np.square(uv_dd_s["interface"]["u"][j]-uv_dd_w["interface"]["u"][j])) +\
          np.sum(np.square(uv_dd_s["interface"]["v"][j]-uv_dd_w["interface"]["v"][j]))
    den = np.sum(np.square(uv_dd_s["interior"]["u"][j])) +\
          np.sum(np.square(uv_dd_s["interior"]["v"][j])) +\
          np.sum(np.square(uv_dd_s["interface"]["u"][j])) +\
          np.sum(np.square(uv_dd_s["interface"]["v"][j]))
    error += num/den
error = np.sqrt(error/mesh_dd.n_sub)
print(f"Error between solutions to strongly and weakly constrained FOMs = {error:1.4e}")

In [ ]:
# plot FD u and v
U_dd_w = uv_dd_w["res"]["u"].reshape(ny, nx)
V_dd_w = uv_dd_w["res"]["v"].reshape(ny, nx)

In [ ]:
plot_kwargs["save"] = True
plot_field(
  z=U_dd_w,
  label="$u_{dd-w}$",
  filename=fig_dir + "/u_dd_weak.png",
  **plot_kwargs
)
plot_field(
  z=V_dd_w,
  label="$v_{dd-w}$",
  filename=fig_dir + "/v_dd_weak.png",
  **plot_kwargs
)

In [ ]:
dd_u_rel_err = np.linalg.norm(uv_dd_s["res"]["u"]-uv["u"].reshape(-1))/np.linalg.norm(uv["u"])
dd_v_rel_err = np.linalg.norm(uv_dd_s["res"]["v"]-uv["v"].reshape(-1))/np.linalg.norm(uv["v"])
print("DD u relative error =", dd_u_rel_err)
print("DD v relative error =", dd_v_rel_err)